# PDF RAG Demo
A step-by-step pipeline: PDF → chunks → embeddings → Chroma → Groq.

Install requirements, configure `.env`, and run these cells in order. Launch the GUI separately with `python -m streamlit run app.py`.

In [ ]:
# Optional: install packages into the current notebook kernel.
# %pip install -r requirements.txt

## 1. Load the pipeline
Use this notebook from the project folder. Credentials are read from `.env` and are never printed.

In [ ]:
from pathlib import Path
from rag_pipeline import extract_chunks, load_embedder, index_pdf, retrieve, answer_question

PDF_PATH = Path("your_document.pdf")  # Change this to your PDF file
QUESTION = "What are the main topics discussed in this document?"

## 2. Read and split the PDF
Each chunk retains its original PDF page number. Image-only PDFs require OCR.

In [ ]:
pdf_bytes = PDF_PATH.read_bytes()
chunks, page_count = extract_chunks(pdf_bytes)
print(f"Pages: {page_count}, chunks: {len(chunks)}")
chunks[0]

## 3. Create local embeddings
The first run downloads the sentence-transformer model. No Groq API call is used for embedding.

In [ ]:
embedder = load_embedder()
example_vector = embedder.encode(chunks[0]["text"], normalize_embeddings=True)
print("Embedding dimensions:", len(example_vector))

## 4. Store vectors in Chroma
The index persists in `chroma_db/`; repeated uploads reuse the same document index.

In [ ]:
collection, page_count, chunk_count = index_pdf(pdf_bytes, embedder)
print("Stored chunks:", collection.count())

## 5. Retrieve relevant excerpts
Cosine distance ranks chunks; lower distance means closer embeddings.

In [ ]:
sources = retrieve(QUESTION, collection, embedder, top_k=4)
for source in sources:
    print(f"Page {source['page']} | distance {source['distance']:.3f}")
    print(source["text"], "\n")

## 6. Ask Groq
This cell sends your question and retrieved excerpts to Groq. Set your API key in `.env` before running it. Verify the answer against the sources.

In [ ]:
answer = answer_question(QUESTION, sources)
print(answer)

## 7. Try another question

In [ ]:
question = "What does the PDF say about its key conclusions?"
sources = retrieve(question, collection, embedder, top_k=4)
print(answer_question(question, sources))